# 01 · Model comparison tuned

Este notebook compara **tres modelos ajustados con `RandomizedSearchCV`** para el problema de churn:

- Logistic Regression
- Random Forest
- Deep Neural Network (tabular, con SciKeras + TensorFlow)

## Objetivos
- definir un flujo homogéneo de modelado
- ajustar hiperparámetros con validación cruzada estratificada
- comparar los mejores resultados de cada familia de modelos
- exportar una tabla final para integrarla en la app de Streamlit

## Nota
Este notebook está pensado como el **notebook comparación de modelos**.  
Después, el mejor modelo se entrenará de forma final en un segundo notebook.


## Imports y configuración

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "churned"

EXPORT_DIR = Path("../data/exports")
DATA_PATH = Path("../data/processed/train_model_ready.csv")


## Dependencias para la red neuronal

In [3]:
# Si esta celda falla, instala antes:
# python -m pip install scikeras tensorflow

import tensorflow as tf
from scikeras.wrappers import KerasClassifier # envuelve la red neural en un wrapper para poder usar RandomizedSearchCV de scikit-learn

print("TensorFlow:", tf.__version__)


2026-05-06 12:57:38.551722: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.16.2


## Carga de datos

In [4]:
df = pd.read_csv(DATA_PATH).copy()

print("Shape:", df.shape)
df.head()


Shape: (125000, 29)


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,...,churned,tenure_days,tenure_months,songs_per_hour,high_skip_user,age_group,weekly_hours_bin,skip_rate_bin,state_code,region
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,...,0,1606,53.533333,7.547554,0,25-34,20-30,0-0.2,MT,West
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,...,1,2897,96.566667,1.877504,1,50-64,20-30,0.8-1.0,NJ,Northeast
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,...,0,348,11.600000,15.843835,0,50-64,10-20,0-0.2,WA,West
3,4,63,California,Family,Yearly,4,Apple Pay,Medium,-2894,22.842084,...,0,2894,96.466667,19.350248,0,50-64,20-30,0-0.2,CA,West
4,5,54,Washington,Family,Monthly,3,Paypal,High,-92,23.151163,...,0,92,3.066667,10.496233,0,50-64,20-30,0-0.2,WA,West


## Preparación del dataset

Eliminamos columnas auxiliares si existen y separamos predictores y objetivo.


In [5]:
drop_cols = [col for col in ["y_true", "y_pred", "p_churn"] if col in df.columns]

model_df = df.drop(columns=drop_cols).copy()

if TARGET not in model_df.columns:
    raise ValueError(f"No se encontró la columna objetivo '{TARGET}'.")

X = model_df.drop(columns=[TARGET]).copy()
y = model_df[TARGET].copy()

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))
print("Shape X:", X.shape)
print("Shape y:", y.shape)


Variables numéricas: 18
Variables categóricas: 10
Shape X: (125000, 28)
Shape y: (125000,)


## Split holdout para evaluación final

El tuning se hará en `train`, y luego usaremos `valid` para evaluar el mejor modelo de cada familia.


In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)


Train shape: (100000, 28)
Valid shape: (25000, 28)


## Preprocesadores

In [7]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)


## Función para la DNN tabular

In [8]:
def build_dnn_model(
    meta,
    hidden_units=(64, 32),
    dropout_rate=0.2,
    learning_rate=0.001
):
    n_features = meta["n_features_in_"]

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(n_features,)))

    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
        if dropout_rate and dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )

    return model


## Definición de pipelines base

In [9]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("classifier", LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE
    ))
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_unscaled),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

dnn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("classifier", KerasClassifier(
        model=build_dnn_model,
        verbose=0,
        random_state=RANDOM_STATE
    ))
])

logreg_pipeline, rf_pipeline, dnn_pipeline


(Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median')),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   ['customer_id', 'age',
                                                    'num_subscription_pauses',
                                                    'signup_date',
                                                    'weekly_hours',
                                                    'average_session_length',
                                                    'song_skip_rate',
                                                    'weekly_songs_played',
                                                

## Espacios de búsqueda

In [10]:
logreg_param_dist = {
    "classifier__C": np.logspace(-3, 2, 12),
    "classifier__class_weight": [None, "balanced"],
    "classifier__solver": ["lbfgs"],
}

rf_param_dist = {
    "classifier__n_estimators": [100, 200, 300, 400],
    "classifier__max_depth": [5, 8, 12, 16, None],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", None],
    "classifier__class_weight": [None, "balanced"],
}

dnn_param_dist = {
    "classifier__model__hidden_units": [(32,), (64, 32), (128, 64)],
    "classifier__model__dropout_rate": [0.0, 0.2, 0.3],
    "classifier__model__learning_rate": [0.001, 0.0005],
    "classifier__batch_size": [32, 64],
    "classifier__epochs": [20, 30],
}


## Validación cruzada y callback para la DNN

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="loss",
    patience=3,
    restore_best_weights=True
)

dnn_fit_kwargs = {
    "classifier__callbacks": [early_stopping]
}


## Función auxiliar para evaluar el mejor modelo de cada familia

In [12]:
def evaluate_best_estimator(name, estimator, X_valid, y_valid):
    y_pred = estimator.predict(X_valid)

    if hasattr(estimator, "predict_proba"):
        y_proba = estimator.predict_proba(X_valid)[:, 1]
    else:
        y_proba = estimator.predict(X_valid)

    row = {
        "model": name,
        "accuracy": accuracy_score(y_valid, y_pred),
        "precision": precision_score(y_valid, y_pred, zero_division=0),
        "recall": recall_score(y_valid, y_pred, zero_division=0),
        "f1": f1_score(y_valid, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_valid, y_proba),
    }
    return row


## RandomizedSearchCV · Logistic Regression

In [13]:
logreg_search = RandomizedSearchCV(
    estimator=logreg_pipeline,
    param_distributions=logreg_param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

logreg_search.fit(X_train, y_train)

print("Mejores parámetros LogReg:")
print(logreg_search.best_params_)
print("Mejor ROC AUC CV LogReg:", round(logreg_search.best_score_, 4))


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Mejores parámetros LogReg:
{'classifier__solver': 'lbfgs', 'classifier__class_weight': None, 'classifier__C': 4.328761281083057}
Mejor ROC AUC CV LogReg: 0.9302


## RandomizedSearchCV · Random Forest

In [14]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

rf_search.fit(X_train, y_train)

print("Mejores parámetros RF:")
print(rf_search.best_params_)
print("Mejor ROC AUC CV RF:", round(rf_search.best_score_, 4))


Fitting 5 folds for each of 20 candidates, totalling 100 fits


KeyboardInterrupt: 

## RandomizedSearchCV · Deep Neural Network

In [ ]:
dnn_search = RandomizedSearchCV(
    estimator=dnn_pipeline,
    param_distributions=dnn_param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

dnn_search.fit(X_train, y_train, **dnn_fit_kwargs)

print("Mejores parámetros DNN:")
print(dnn_search.best_params_)
print("Mejor ROC AUC CV DNN:", round(dnn_search.best_score_, 4))


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Mejores parámetros DNN:
{'classifier__model__learning_rate': 0.001, 'classifier__model__hidden_units': (128, 64), 'classifier__model__dropout_rate': 0.0, 'classifier__epochs': 20, 'classifier__batch_size': 32}
Mejor ROC AUC CV DNN: nan


## Comparación de los mejores modelos ajustados

In [ ]:
comparison_rows = []

comparison_rows.append(evaluate_best_estimator("LogReg Tuned", logreg_search.best_estimator_, X_valid, y_valid))
comparison_rows.append(evaluate_best_estimator("RF Tuned", rf_search.best_estimator_, X_valid, y_valid))
comparison_rows.append(evaluate_best_estimator("DNN Tuned", dnn_search.best_estimator_, X_valid, y_valid))

comparison_df = pd.DataFrame(comparison_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
comparison_df


,model,accuracy,precision,recall,f1,roc_auc
0,RF Tuned,0.84728,0.846408,0.858278,0.852302,0.938177
1,LogReg Tuned,0.84160,0.846166,0.845111,0.845638,0.930247
2,DNN Tuned,0.83184,0.852833,0.812700,0.832283,0.922399


## Visualización comparativa

In [ ]:
comparison_long = comparison_df.melt(
    id_vars="model",
    value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"],
    var_name="metric",
    value_name="score"
)

metric_order = ["accuracy", "precision", "recall", "f1", "roc_auc"]
comparison_long["metric"] = pd.Categorical(comparison_long["metric"], categories=metric_order, ordered=True)
comparison_long = comparison_long.sort_values(["metric", "model"])

fig = px.bar(
    comparison_long,
    x="metric",
    y="score",
    color="model",
    barmode="group",
    text="score",
    title="Comparación de modelos tuneados"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    title_x=0.5,
    xaxis_title="Métrica",
    yaxis_title="Score"
)

fig.show()


## Tabla de mejores hiperparámetros

In [ ]:
best_params_df = pd.DataFrame([
    {
        "model": "LogReg Tuned",
        **logreg_search.best_params_
    },
    {
        "model": "RF Tuned",
        **rf_search.best_params_
    },
    {
        "model": "DNN Tuned",
        **dnn_search.best_params_
    },
])

best_params_df


,model,classifier__solver,classifier__class_weight,classifier__C,classifier__n_estimators,classifier__min_samples_split,classifier__min_samples_leaf,classifier__max_features,classifier__max_depth,classifier__model__learning_rate,classifier__model__hidden_units,classifier__model__dropout_rate,classifier__epochs,classifier__batch_size
0,LogReg Tuned,lbfgs,NaN,4.328761,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RF Tuned,NaN,NaN,NaN,400.0,2.0,5.0,sqrt,16.0,NaN,NaN,NaN,NaN,NaN
2,DNN Tuned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001,"(128, 64)",0.0,20.0,32.0


## Tabla de métricas, coeficientes y exportación para Regresión logística y DNN

In [ ]:
def export_model_outputs(estimator, model_name, X_valid, y_valid, export_prefix):
    y_pred = estimator.predict(X_valid)

    if hasattr(estimator, "predict_proba"):
        y_proba = estimator.predict_proba(X_valid)[:, 1]
    else:
        y_proba = estimator.predict(X_valid)

    metrics_df = pd.DataFrame([{
        "model": model_name,
        "accuracy": accuracy_score(y_valid, y_pred),
        "precision": precision_score(y_valid, y_pred, zero_division=0),
        "recall": recall_score(y_valid, y_pred, zero_division=0),
        "f1": f1_score(y_valid, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_valid, y_proba),
    }])

    preds_df = X_valid.copy()
    preds_df["y_true"] = y_valid.values
    preds_df["y_pred"] = y_pred
    preds_df["p_churn"] = y_proba
    preds_df = preds_df.sort_values("p_churn", ascending=False).reset_index(drop=True)

    cm = confusion_matrix(y_valid, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    cm_pct_df = pd.DataFrame(
        cm_pct,
        index=["Actual No Churn", "Actual Churn"],
        columns=["Pred No Churn", "Pred Churn"]
    )

    metrics_df.to_csv(EXPORT_DIR / f"{export_prefix}_metrics.csv", index=False)
    preds_df.to_csv(EXPORT_DIR / f"{export_prefix}_validation_predictions.csv", index=False)
    cm_pct_df.to_csv(EXPORT_DIR / f"{export_prefix}_confusion_matrix_percentage.csv")

    print(f"Exportado: {export_prefix}")
    print("-", EXPORT_DIR / f"{export_prefix}_metrics.csv")
    print("-", EXPORT_DIR / f"{export_prefix}_validation_predictions.csv")
    print("-", EXPORT_DIR / f"{export_prefix}_confusion_matrix_percentage.csv")

    return metrics_df, preds_df, cm_pct_df

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

logreg_tuned_metrics_df, logreg_tuned_preds_df, logreg_tuned_cm_pct_df = export_model_outputs(
    estimator=logreg_search.best_estimator_,
    model_name="LogReg Tuned",
    X_valid=X_valid,
    y_valid=y_valid,
    export_prefix="logreg_tuned"
)

Exportado: logreg_tuned
- ../data/exports/logreg_tuned_metrics.csv
- ../data/exports/logreg_tuned_validation_predictions.csv
- ../data/exports/logreg_tuned_confusion_matrix_percentage.csv


In [ ]:
dnn_tuned_metrics_df, dnn_tuned_preds_df, dnn_tuned_cm_pct_df = export_model_outputs(
    estimator=dnn_search.best_estimator_,
    model_name="DNN Tuned",
    X_valid=X_valid,
    y_valid=y_valid,
    export_prefix="dnn_tuned"
)

Exportado: dnn_tuned
- ../data/exports/dnn_tuned_metrics.csv
- ../data/exports/dnn_tuned_validation_predictions.csv
- ../data/exports/dnn_tuned_confusion_matrix_percentage.csv


In [ ]:
logreg_best = logreg_search.best_estimator_

feature_names = logreg_best.named_steps["preprocessor"].get_feature_names_out()
coefs = logreg_best.named_steps["classifier"].coef_[0]

logreg_coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs
}).sort_values("coefficient", ascending=False).reset_index(drop=True)

logreg_coef_df["feature_clean"] = (
    logreg_coef_df["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ", regex=False)
)

logreg_coef_df.head(20)

,feature,coefficient,feature_clean
0,cat__weekly_hours_bin_0-5,3.463320,weekly hours bin 0-5
1,cat__subscription_type_Free,2.625344,subscription type Free
2,cat__customer_service_inquiries_High,2.129635,customer service inquiries High
3,cat__age_group_18-24,1.974952,age group 18-24
4,cat__weekly_hours_bin_5-10,1.383362,weekly hours bin 5-10
5,num__high_skip_user,0.941494,high skip user
6,num__num_subscription_pauses,0.869263,num subscription pauses
7,num__age,0.859711,age
8,cat__age_group_25-34,0.576970,age group 25-34
9,cat__subscription_type_Student,0.567102,subscription type Student


In [ ]:
logreg_coef_df.to_csv(EXPORT_DIR / "logreg_tuned_coefficients.csv", index=False)

print(EXPORT_DIR / "logreg_tuned_coefficients.csv")

../data/exports/logreg_tuned_coefficients.csv


### Extra: curvas para explicar DNN

In [ ]:
# Curvas de entrenamiento de la mejor DNN
# Reentrena la configuración ganadora para visualizar loss y AUC por época

import matplotlib.pyplot as plt
import tensorflow as tf

# 1) Mejores hiperparámetros de la DNN encontrados en RandomizedSearchCV
best_dnn_params = dnn_search.best_params_
print(best_dnn_params)

hidden_units = best_dnn_params["classifier__model__hidden_units"]
dropout_rate = best_dnn_params["classifier__model__dropout_rate"]
learning_rate = best_dnn_params["classifier__model__learning_rate"]
batch_size = best_dnn_params["classifier__batch_size"]
epochs = best_dnn_params["classifier__epochs"]

# 2) Preprocesado igual que en la DNN del pipeline
X_train_prepared = preprocessor_scaled.fit_transform(X_train)
X_valid_prepared = preprocessor_scaled.transform(X_valid)

print("Shapes:", X_train_prepared.shape, X_valid_prepared.shape)

# 3) Función para construir la red final
def build_final_dnn(n_features, hidden_units=(64, 32), dropout_rate=0.2, learning_rate=0.001):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(n_features,)))

    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
        if dropout_rate and dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )

    return model

# 4) Construcción de la mejor DNN
n_features = X_train_prepared.shape[1]

final_dnn = build_final_dnn(
    n_features=n_features,
    hidden_units=hidden_units,
    dropout_rate=dropout_rate,
    learning_rate=learning_rate
)

final_dnn.summary()

# 5) Early stopping para evitar entrenar de más
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# 6) Entrenamiento guardando history
history = final_dnn.fit(
    X_train_prepared,
    y_train,
    validation_data=(X_valid_prepared, y_valid),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping],
    verbose=1
)

print(history.history.keys())

# 7) Curva de loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("DNN - Loss por época")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 8) Curva de AUC
plt.figure(figsize=(8, 5))
plt.plot(history.history["auc"], label="Train AUC")
plt.plot(history.history["val_auc"], label="Validation AUC")
plt.title("DNN - AUC por época")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 9) Guardado opcional de figuras
# plt.figure(figsize=(8, 5))
# plt.plot(history.history["loss"], label="Train Loss")
# plt.plot(history.history["val_loss"], label="Validation Loss")
# plt.title("DNN - Loss por época")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.savefig("data/exports/dnn_loss_curve.png", dpi=200, bbox_inches="tight")
# plt.show()

# plt.figure(figsize=(8, 5))
# plt.plot(history.history["auc"], label="Train AUC")
# plt.plot(history.history["val_auc"], label="Validation AUC")
# plt.title("DNN - AUC por época")
# plt.xlabel("Epoch")
# plt.ylabel("AUC")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.savefig("data/exports/dnn_auc_curve.png", dpi=200, bbox_inches="tight")
# plt.show()